# Customer propensity and upsell scoring — exploratory data analysis

This notebook explores the marketing campaign dataset to understand response patterns,
customer segments, and features that drive upsell propensity.

## 1. The business question

Telecom companies run upsell campaigns to move customers from lower-tier to higher-tier plans. Mass campaigns are expensive and yield low conversion rates. The question: **can we predict which customers are most likely to respond**, so marketing can focus resources on the best prospects?

We start by loading the campaign dataset and exploring the response patterns that will inform our feature engineering and modeling choices.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/marketing_campaign.csv')
print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## 2. Response rate by segment

With only ~12% of customers responding, this is an imbalanced classification problem. Before building models, we need to understand which segments respond at higher rates. The plots below break down response rate by plan type, income, tenure, usage patterns, and channel preference.

In [ ]:
resp_rate = df['responded'].mean()
print(f'Overall response rate: {resp_rate:.2%} ({df["responded"].sum()} / {len(df)})')

fig, ax = plt.subplots(figsize=(5, 4))
df['responded'].value_counts().plot.bar(ax=ax, color=['#90CAF9', '#2196F3'])
ax.set_title('Response distribution')
ax.set_xticklabels(['No response', 'Responded'], rotation=0)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### Response rate by current plan

Standard plan customers have the most upsell headroom and the highest response rate. Premium customers, already on the top tier, have nowhere to upgrade.

In [ ]:
plan_resp = df.groupby('current_plan')['responded'].agg(['mean', 'sum', 'count'])
plan_resp.columns = ['Response rate', 'Responders', 'Total']
plan_resp = plan_resp.loc[['Basic', 'Standard', 'Premium']]
print(plan_resp)

fig, ax = plt.subplots(figsize=(6, 4))
plan_resp['Response rate'].plot.bar(ax=ax, color='#2196F3')
ax.set_ylabel('Response rate')
ax.set_title('Response rate by current plan')
ax.set_xticklabels(plan_resp.index, rotation=0)
for i, v in enumerate(plan_resp['Response rate']):
    ax.text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Income distribution by response

Higher income customers may have more willingness to pay for premium tiers. Let's compare the income distribution between responders and non-responders.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for resp, label, color in [(0, 'No response', '#90CAF9'), (1, 'Responded', '#1565C0')]:
    subset = df[df['responded'] == resp]['income']
    ax.hist(subset, bins=40, alpha=0.6, label=label, color=color, density=True)
ax.set_xlabel('Annual income ($)')
ax.set_ylabel('Density')
ax.set_title('Income distribution by response')
ax.legend()
plt.tight_layout()
plt.show()

print('Median income:')
print(df.groupby('responded')['income'].median())

### Tenure and response

Loyal customers with longer tenure may be more receptive to upsell offers because they already trust the brand.

In [ ]:
df['tenure_bucket'] = pd.cut(df['tenure_months'], bins=[0, 12, 24, 48, 72, 120],
                              labels=['0-12', '13-24', '25-48', '49-72', '73-120'])

tenure_resp = df.groupby('tenure_bucket')['responded'].mean()
fig, ax = plt.subplots(figsize=(7, 4))
tenure_resp.plot.bar(ax=ax, color='#2196F3')
ax.set_ylabel('Response rate')
ax.set_title('Response rate by tenure bucket')
ax.set_xticklabels(tenure_resp.index, rotation=0)
plt.tight_layout()
plt.show()

### Usage patterns

Heavy users of data, calls, and SMS may be outgrowing their current plan, making them natural candidates for an upgrade.

In [ ]:
usage_cols = ['data_usage_gb', 'call_minutes', 'sms_count', 'monthly_spend']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), usage_cols):
    for resp, label, color in [(0, 'No response', '#90CAF9'), (1, 'Responded', '#1565C0')]:
        subset = df[df['responded'] == resp][col]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=8)
fig.suptitle('Usage patterns by response', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Channel preference and previous upsell response

Previous campaign behavior is often the strongest predictor of future behavior. Channel preference may also matter if certain channels are more persuasive.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Channel preference
ch_resp = df.groupby('channel_preference')['responded'].mean().sort_values(ascending=False)
ch_resp.plot.bar(ax=axes[0], color='#2196F3')
axes[0].set_title('Response rate by channel preference')
axes[0].set_ylabel('Response rate')
axes[0].set_xticklabels(ch_resp.index, rotation=30)

# Previous upsell response
prev_resp = df.groupby('previous_upsell_response')['responded'].mean()
prev_resp.plot.bar(ax=axes[1], color='#2196F3')
axes[1].set_title('Response rate by previous upsell response')
axes[1].set_xticklabels(['No prev response', 'Prev responded'], rotation=0)
axes[1].set_ylabel('Response rate')

plt.tight_layout()
plt.show()

## 3. Feature engineering insights

The correlation heatmap below reveals which raw features correlate with the response variable, guiding the feature engineering in `src/data_loader.py`. Key engineered features include:
- **revenue_per_tenure** — how much a customer spends relative to their loyalty
- **usage_intensity** — normalized composite of data, calls, and SMS
- **upsell_headroom** — how many plan levels remain to upgrade
- **income_x_tenure** and **data_x_streaming** — interaction terms

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation matrix')
plt.tight_layout()
plt.show()

## 4. Key takeaways

- **Standard plan customers are the prime upsell target**: they have room to upgrade and show the highest response rate, which motivates the `upsell_headroom` feature.
- **Behavioral signals outweigh demographics**: previous upsell response, data usage, and channel preference are stronger predictors than age or income alone, suggesting interaction features will add value.
- **The ~12% response rate means targeting matters**: contacting everyone wastes 88% of the budget on non-responders. A well-calibrated propensity model that ranks customers into deciles can concentrate spend on the top 30% and still capture the majority of responders.

The full model pipeline (Logistic Regression, Random Forest, XGBoost with calibration and decile analysis) is implemented in `src/model.py`. Running it produces lift charts, calibration curves, and a campaign ROI comparison showing that targeted outreach achieves over 2,000% ROI compared to mass campaigns.